# SGLang 显存优化技巧 — 低显存显卡适配方法

**定位：** 面向使用消费级 GPU（如 RTX 3060 6GB、RTX 4060 8GB）的开发者，详解在有限显存条件下高效运行 SGLang 的实用技巧。

**服务端启动命令（默认示例）：**

```bash
python -m sglang.launch_server --model-path Qwen/Qwen2.5-0.5B-Instruct --host 127.0.0.1 --port 30000
```

> 本 Notebook 为原创示例代码，可自由用于学习、教学及发布到 Gitee 等平台

---

## 硬件与环境要求

| 项目 | 最低要求 |
|------|----------|
| GPU | NVIDIA GPU（支持 CUDA） |
| 显存（VRAM） | ≥ 6 GB（推荐 8 GB 以上） |
| 驱动 | NVIDIA Driver ≥ 525 |
| CUDA | CUDA ≥ 12.1 |
| Python | ≥ 3.9 |
| 操作系统 | Linux / WSL2 推荐 |

## 依赖安装

请确保已安装以下依赖：

```bash
pip install "sglang[all]" "openai>=1.0.0" "requests>=2.28.0"
```

如需 PyTorch 支持（通常已包含），可单独安装：

```bash
pip install torch
```

---

## 使用指南

1. 运行「环境检查」单元格，确认 GPU 和依赖就绪
2. 按需运行「可选安装」单元格
3. 依次运行各显存优化技巧的演示单元格
4. 最后运行「清理」单元格释放资源

---

In [ ]:
# ============================================================
# 环境检查 —— 确认 GPU、PyTorch 和 SGLang 是否就绪
# ============================================================

import sys  # 导入系统模块，用于获取 Python 版本信息

print("Python 版本：", sys.version)  # 打印当前 Python 版本

# ---------- 检查 PyTorch ----------
try:  # 尝试导入 PyTorch
    import torch  # 导入 PyTorch 库
    print("PyTorch 版本：", torch.__version__)  # 打印 PyTorch 版本号
    print("CUDA 是否可用：", torch.cuda.is_available())  # 检查 CUDA 是否可用
    if torch.cuda.is_available():  # 如果 CUDA 可用
        print("GPU 名称：", torch.cuda.get_device_name(0))  # 打印第一块 GPU 名称
        vram_total = torch.cuda.get_device_properties(0).total_mem / 1024**3  # 计算总显存（GB）
        print(f"GPU 总显存：{vram_total:.1f} GB")  # 打印总显存大小
except ImportError:  # 如果 PyTorch 未安装
    print("⚠ PyTorch 未安装，请先安装 PyTorch")  # 提示用户安装

# ---------- 检查 SGLang ----------
try:  # 尝试导入 sglang
    import sglang  # 导入 sglang 库
    print("SGLang 版本：", sglang.__version__)  # 打印 SGLang 版本号
except ImportError:  # 如果 SGLang 未安装
    print("⚠ SGLang 未安装，请运行下方安装单元格")  # 提示用户安装

print("\n环境检查完成 ✓")  # 打印检查完成提示

In [ ]:
# ============================================================
# 可选安装 —— 如依赖缺失则取消注释执行
# ============================================================

# !pip install "sglang[all]" "openai>=1.0.0" "requests>=2.28.0"  # 安装 SGLang 全部依赖及 OpenAI 客户端和 requests 库

## 为什么需要显存优化？

大语言模型（LLM）在推理时需要大量显存（VRAM），主要用于存放：

- **模型权重**：参数量越大，占用越多
- **KV Cache**：上下文越长，缓存越大
- **运行时临时张量**：计算过程中产生的中间数据

对于消费级 GPU（6GB-8GB 显存），直接使用默认配置常常会遇到 **CUDA Out of Memory** 错误。

本 Notebook 将介绍 **四种实用技巧**，帮助你在有限显存下顺利运行 SGLang。

---

## 启动前：显存状态检查

在进行任何优化之前，先了解当前 GPU 显存使用情况。

In [ ]:
# ============================================================
# 显存状态检查 —— 查看 GPU 显存的详细使用情况
# ============================================================

import subprocess  # 导入子进程模块，用于执行系统命令

def check_gpu_memory():  # 定义检查显存的函数
    """通过 nvidia-smi 和 PyTorch 两种方式检查 GPU 显存。"""

    # ---------- 方法一：通过 nvidia-smi 命令行工具 ----------
    print("=" * 50)  # 打印分隔线
    print("方法一：nvidia-smi 输出")  # 打印标题
    print("=" * 50)  # 打印分隔线
    try:  # 尝试执行 nvidia-smi 命令
        result = subprocess.run(  # 运行子进程命令
            ["nvidia-smi",  # 调用 nvidia-smi 工具
             "--query-gpu=name,memory.total,memory.used,memory.free,utilization.gpu",  # 查询 GPU 信息
             "--format=csv,noheader,nounits"],  # 输出为 CSV 格式
            capture_output=True,  # 捕获标准输出
            text=True  # 以文本模式返回
        )
        if result.returncode == 0:  # 如果命令执行成功
            parts = result.stdout.strip().split(",")  # 按逗号分割输出
            parts = [p.strip() for p in parts]  # 去除首尾空格
            print(f"  GPU 名称   ：{parts[0]}")  # 打印 GPU 名称
            print(f"  总显存     ：{parts[1]} MB")  # 打印总显存
            print(f"  已用显存   ：{parts[2]} MB")  # 打印已使用显存
            print(f"  空闲显存   ：{parts[3]} MB")  # 打印空闲显存
            print(f"  GPU 利用率 ：{parts[4]}%")  # 打印 GPU 利用率
        else:  # 如果命令执行失败
            print("  nvidia-smi 执行失败")  # 提示失败
    except FileNotFoundError:  # 如果找不到 nvidia-smi
        print("  nvidia-smi 未找到，可能未安装 NVIDIA 驱动")  # 提示未安装驱动

    # ---------- 方法二：通过 PyTorch CUDA API ----------
    print("\n" + "=" * 50)  # 打印分隔线
    print("方法二：PyTorch CUDA API")  # 打印标题
    print("=" * 50)  # 打印分隔线
    try:  # 尝试使用 PyTorch 查询显存
        import torch  # 导入 PyTorch
        if torch.cuda.is_available():  # 如果 CUDA 可用
            total = torch.cuda.get_device_properties(0).total_mem / 1024**3  # 获取总显存（GB）
            reserved = torch.cuda.memory_reserved(0) / 1024**3  # 获取已预留显存（GB）
            allocated = torch.cuda.memory_allocated(0) / 1024**3  # 获取已分配显存（GB）
            print(f"  总显存     ：{total:.2f} GB")  # 打印总显存
            print(f"  已预留     ：{reserved:.2f} GB")  # 打印已预留显存
            print(f"  已分配     ：{allocated:.2f} GB")  # 打印已分配显存
            print(f"  可用估算   ：{total - reserved:.2f} GB")  # 打印可用显存
        else:  # 如果 CUDA 不可用
            print("  CUDA 不可用")  # 提示不可用
    except ImportError:  # 如果 PyTorch 未安装
        print("  PyTorch 未安装")  # 提示未安装

check_gpu_memory()  # 调用函数执行检查

## 技巧一：调整 `--mem-fraction-static`

SGLang 默认会预分配 **88%** 的可用显存用于 KV Cache（`--mem-fraction-static 0.88`）。

在低显存显卡上，这可能导致模型加载后剩余空间不足而 OOM。

**解决方案：** 降低该比例，为模型权重和运行时张量留出更多空间。

| 参数值 | 含义 | 适用场景 |
|--------|------|----------|
| 0.88（默认） | 88% 显存用于 KV Cache | 显存充足（≥24GB） |
| 0.70 | 70% 显存用于 KV Cache | 中等显存（8-16GB） |
| 0.50 | 50% 显存用于 KV Cache | 低显存（6-8GB） |

---

In [ ]:
# ============================================================
# 显存分配比例调优 —— 使用 --mem-fraction-static 参数
# ============================================================

import subprocess  # 导入子进程模块
import time  # 导入时间模块
import requests  # 导入 HTTP 请求库

# ---------- 构建优化后的启动命令 ----------
cmd_conservative = [  # 定义保守模式启动命令列表
    "python", "-m", "sglang.launch_server",  # 使用 Python 模块方式启动 SGLang 服务
    "--model-path", "Qwen/Qwen2.5-0.5B-Instruct",  # 指定模型路径
    "--host", "127.0.0.1",  # 绑定本地地址
    "--port", "30000",  # 指定端口号
    "--mem-fraction-static", "0.7",  # 将显存预分配比例从默认 0.88 降低到 0.7
]

# ---------- 打印命令供参考 ----------
print("优化后的启动命令：")  # 打印提示文字
print(" ".join(cmd_conservative))  # 将命令列表拼接为字符串并打印

print("\n与默认配置对比：")  # 打印对比标题
print(f"  默认 --mem-fraction-static = 0.88 → 88% 显存用于 KV Cache")  # 说明默认值
print(f"  优化 --mem-fraction-static = 0.70 → 70% 显存用于 KV Cache")  # 说明优化值
print(f"  节省约 18% 显存用于模型权重和计算")  # 说明节省效果

# ---------- 启动服务（后台运行） ----------
print("\n正在启动 SGLang 服务（保守显存模式）...")  # 打印启动提示
process = subprocess.Popen(  # 以子进程方式启动服务
    cmd_conservative,  # 传入命令列表
    stdout=subprocess.PIPE,  # 捕获标准输出
    stderr=subprocess.PIPE  # 捕获标准错误
)

# ---------- 等待服务就绪 ----------
max_wait = 120  # 最大等待时间设为 120 秒
start_time = time.time()  # 记录开始等待的时间
server_ready = False  # 初始化服务就绪标志为 False

while time.time() - start_time < max_wait:  # 在最大等待时间内循环检查
    try:  # 尝试发送健康检查请求
        resp = requests.get("http://127.0.0.1:30000/health", timeout=3)  # 向健康检查端点发送 GET 请求
        if resp.status_code == 200:  # 如果返回状态码为 200
            server_ready = True  # 将服务就绪标志设为 True
            break  # 跳出循环
    except requests.ConnectionError:  # 如果连接被拒绝
        pass  # 忽略错误继续等待
    time.sleep(3)  # 每隔 3 秒重试一次

if server_ready:  # 如果服务成功启动
    elapsed = time.time() - start_time  # 计算启动耗时
    print(f"服务启动成功！耗时 {elapsed:.1f} 秒")  # 打印成功信息和耗时
    print("\n启动后显存状态：")  # 打印显存状态标题
    check_gpu_memory()  # 调用之前定义的函数检查显存
else:  # 如果服务启动超时
    print(f"服务在 {max_wait} 秒内未就绪，请检查日志")  # 打印超时提示

## 技巧二：限制上下文长度 `--context-length`

KV Cache 的大小与上下文长度（context length）成正比。模型默认的最大上下文长度可能很大（如 32768 tokens），但实际应用中你可能只需要较短的上下文。

**降低上下文长度 → 减少 KV Cache 显存占用。**

| 上下文长度 | KV Cache 相对大小 | 适用场景 |
|------------|-------------------|----------|
| 32768（默认） | 100% | 长文档处理 |
| 4096 | ~12.5% | 一般对话 |
| 2048 | ~6.25% | 短对话、问答 |

---

In [ ]:
# ============================================================
# 上下文长度与显存关系 —— 使用 --context-length 参数限制上下文
# ============================================================

# ---------- 先停止上一个服务 ----------
if 'process' in dir() and process.poll() is None:  # 如果上一个服务进程仍在运行
    process.terminate()  # 发送终止信号
    process.wait(timeout=10)  # 等待进程结束，最多 10 秒
    print("已停止上一个 SGLang 服务")  # 打印停止提示
    time.sleep(3)  # 等待 3 秒确保端口释放

# ---------- 使用短上下文启动 ----------
cmd_short_ctx = [  # 定义短上下文启动命令
    "python", "-m", "sglang.launch_server",  # 使用 Python 模块方式启动
    "--model-path", "Qwen/Qwen2.5-0.5B-Instruct",  # 指定模型路径
    "--host", "127.0.0.1",  # 绑定本地地址
    "--port", "30000",  # 指定端口号
    "--mem-fraction-static", "0.7",  # 降低显存预分配比例
    "--context-length", "2048",  # 将上下文长度限制为 2048 tokens
]

print("短上下文启动命令：")  # 打印提示
print(" ".join(cmd_short_ctx))  # 打印拼接后的命令字符串

print("\n上下文长度对比：")  # 打印对比标题
print("  默认上下文   ：模型最大值（如 32768 tokens）")  # 说明默认值
print("  优化上下文   ：2048 tokens")  # 说明优化值
print("  预期显存节省 ：KV Cache 减少约 93.75%")  # 说明节省效果

# ---------- 启动服务 ----------
print("\n正在启动 SGLang 服务（短上下文模式）...")  # 打印启动提示
process = subprocess.Popen(  # 以子进程方式启动服务
    cmd_short_ctx,  # 传入命令列表
    stdout=subprocess.PIPE,  # 捕获标准输出
    stderr=subprocess.PIPE  # 捕获标准错误
)

# ---------- 等待服务就绪 ----------
start_time = time.time()  # 记录开始等待的时间
server_ready = False  # 初始化就绪标志

while time.time() - start_time < max_wait:  # 在最大等待时间内循环
    try:  # 尝试健康检查
        resp = requests.get("http://127.0.0.1:30000/health", timeout=3)  # 发送健康检查请求
        if resp.status_code == 200:  # 如果响应正常
            server_ready = True  # 标记服务就绪
            break  # 跳出循环
    except requests.ConnectionError:  # 如果连接失败
        pass  # 继续等待
    time.sleep(3)  # 每 3 秒检查一次

if server_ready:  # 如果服务就绪
    elapsed = time.time() - start_time  # 计算启动耗时
    print(f"服务启动成功！耗时 {elapsed:.1f} 秒")  # 打印启动成功信息
    print("\n短上下文模式下显存状态：")  # 打印显存状态标题
    check_gpu_memory()  # 调用显存检查函数
else:  # 如果服务启动失败
    print(f"服务在 {max_wait} 秒内未就绪")  # 打印超时提示

## 技巧三：使用量化模型

量化（Quantization）通过降低模型权重的精度来减少显存占用：

- **FP16**：每个参数 2 字节 → 0.5B 模型约 1GB
- **INT8**：每个参数 1 字节 → 显存减半
- **INT4（AWQ/GPTQ）**：每个参数 0.5 字节 → 显存减少 75%

SGLang 支持加载 **AWQ** 和 **GPTQ** 格式的预量化模型。

---

In [ ]:
# ============================================================
# 量化模型部署 —— 使用 AWQ/GPTQ 量化模型节省显存
# ============================================================

# ---------- 先停止当前服务 ----------
if 'process' in dir() and process.poll() is None:  # 如果当前服务进程仍在运行
    process.terminate()  # 发送终止信号
    process.wait(timeout=10)  # 等待进程结束
    print("已停止当前 SGLang 服务")  # 打印停止提示

# ---------- AWQ 量化模型启动命令 ----------
cmd_awq = [  # 定义 AWQ 量化模型的启动命令
    "python", "-m", "sglang.launch_server",  # 使用 Python 模块方式启动
    "--model-path", "Qwen/Qwen2.5-0.5B-Instruct-AWQ",  # 指定 AWQ 量化版模型路径
    "--host", "127.0.0.1",  # 绑定本地地址
    "--port", "30000",  # 指定端口号
    "--quantization", "awq",  # 显式指定量化方式为 AWQ
]

# ---------- GPTQ 量化模型启动命令（供参考） ----------
cmd_gptq = [  # 定义 GPTQ 量化模型的启动命令
    "python", "-m", "sglang.launch_server",  # 使用 Python 模块方式启动
    "--model-path", "Qwen/Qwen2.5-0.5B-Instruct-GPTQ-Int4",  # 指定 GPTQ 量化版模型路径
    "--host", "127.0.0.1",  # 绑定本地地址
    "--port", "30000",  # 指定端口号
    "--quantization", "gptq",  # 显式指定量化方式为 GPTQ
]

print("AWQ 量化模型启动命令：")  # 打印 AWQ 命令标题
print(" ".join(cmd_awq))  # 打印拼接后的 AWQ 命令

print("\nGPTQ 量化模型启动命令：")  # 打印 GPTQ 命令标题
print(" ".join(cmd_gptq))  # 打印拼接后的 GPTQ 命令

print("\n显存对比估算（以 0.5B 模型为例）：")  # 打印对比标题
print("  FP16 原始模型 ：约 1.0 GB")  # FP16 模型的显存占用
print("  AWQ INT4 量化 ：约 0.3 GB")  # AWQ 量化后的显存占用
print("  GPTQ INT4 量化：约 0.3 GB")  # GPTQ 量化后的显存占用
print("  显存节省     ：约 70%")  # 量化节省的显存比例

print("\n提示：量化模型需要使用已预量化的模型文件")  # 提示用户需预量化模型
print("  可在 HuggingFace 搜索模型名 + AWQ/GPTQ 关键词")  # 告知搜索方法
print("  例如：Qwen/Qwen2.5-0.5B-Instruct-AWQ")  # 给出示例模型名
print("\n注意：此单元格仅打印命令，未实际启动服务")  # 说明此单元格仅作展示
print("  如需运行，请确保对应量化模型已下载")  # 提示用户需先下载模型

## 技巧四：选择合适的数据类型 `--dtype`

SGLang 支持通过 `--dtype` 参数指定计算时使用的数据类型：

| 数据类型 | 大小 | 特点 |
|----------|------|------|
| float32 | 4 字节 | 精度最高，显存占用最大（通常不用于推理） |
| float16 | 2 字节 | 通用选择，绝大多数 GPU 支持 |
| bfloat16 | 2 字节 | 数值范围更大，Ampere+ 架构 GPU 支持 |
| half | 2 字节 | 等同于 float16 |

**建议：** 消费级 GPU（RTX 30/40 系列）直接使用 `float16` 即可。

---

In [ ]:
# ============================================================
# 数据类型对显存的影响 —— 对比不同 dtype 的特性
# ============================================================

import torch  # 导入 PyTorch

# ---------- 检测 GPU 对各数据类型的支持情况 ----------
print("当前 GPU 数据类型支持检测：")  # 打印标题
print("=" * 50)  # 打印分隔线

if torch.cuda.is_available():  # 如果 CUDA 可用
    gpu_name = torch.cuda.get_device_name(0)  # 获取 GPU 名称
    capability = torch.cuda.get_device_capability(0)  # 获取 GPU 计算能力
    print(f"GPU: {gpu_name}")  # 打印 GPU 名称
    print(f"计算能力: {capability[0]}.{capability[1]}")  # 打印计算能力版本号

    supports_bf16 = capability[0] >= 8  # 计算能力 >= 8.0 才支持 bfloat16
    print(f"\nfloat16  支持：是（所有 CUDA GPU 均支持）")  # float16 所有 CUDA GPU 都支持
    bf16_status = "是" if supports_bf16 else "否"  # 根据计算能力判断 bfloat16 支持状态
    print(f"bfloat16 支持：{bf16_status}（需要 Ampere 架构及以上）")  # 打印 bfloat16 支持状态

    # ---------- 不同 dtype 的启动命令示例 ----------
    print("\n不同数据类型的启动命令：")  # 打印命令示例标题
    base_cmd = "python -m sglang.launch_server --model-path Qwen/Qwen2.5-0.5B-Instruct --host 127.0.0.1 --port 30000"  # 基础启动命令
    print(f"  float16  : {base_cmd} --dtype float16")  # 打印 float16 启动命令
    print(f"  bfloat16 : {base_cmd} --dtype bfloat16")  # 打印 bfloat16 启动命令

    if supports_bf16:  # 如果支持 bfloat16
        print("\n推荐：您的 GPU 支持 bfloat16，可使用 --dtype bfloat16")  # 推荐 bfloat16
    else:  # 如果不支持
        print("\n推荐：您的 GPU 不支持 bfloat16，请使用 --dtype float16")  # 推荐 float16

    # ---------- 演示显存占用差异 ----------
    print("\n数据类型与显存占用（以 0.5B 参数量为例）：")  # 打印占用对比标题
    params_billion = 0.5  # 模型参数量为 0.5B
    params = params_billion * 1e9  # 将参数量转换为实际数字
    fp32_mem = params * 4 / 1024**3  # float32 每个参数 4 字节
    fp16_mem = params * 2 / 1024**3  # float16 每个参数 2 字节
    print(f"  float32 : {fp32_mem:.2f} GB（仅存放权重）")  # 打印 float32 权重显存
    print(f"  float16 : {fp16_mem:.2f} GB（仅存放权重）")  # 打印 float16 权重显存
    print(f"  bfloat16: {fp16_mem:.2f} GB（与 float16 相同大小）")  # 打印 bfloat16 权重显存
else:  # 如果 CUDA 不可用
    print("CUDA 不可用，无法检测 GPU 数据类型支持")  # 提示不可用

## 显存优化总结表

| 技巧 | 参数 | 预期效果 | 适用场景 |
|------|------|----------|----------|
| 降低显存预分配比例 | `--mem-fraction-static 0.7` | 减少 KV Cache 预分配，留更多空间给模型 | 6-8GB 显存显卡 |
| 限制上下文长度 | `--context-length 2048` | 大幅减少 KV Cache 显存占用 | 短对话、问答场景 |
| 使用量化模型 | `--quantization awq` | 模型权重显存减少 50-75% | 对精度要求不高的场景 |
| 选择合适数据类型 | `--dtype float16` | 确保兼容性，避免 bfloat16 不兼容问题 | 消费级 GPU |

**组合使用效果最佳：**

```bash
python -m sglang.launch_server \
    --model-path Qwen/Qwen2.5-0.5B-Instruct \
    --host 127.0.0.1 --port 30000 \
    --mem-fraction-static 0.7 \
    --context-length 2048 \
    --dtype float16
```

---

In [ ]:
# ============================================================
# 清理 —— 停止 SGLang 服务并释放资源
# ============================================================

import subprocess  # 导入子进程模块

# ---------- 停止 SGLang 服务进程 ----------
if 'process' in dir() and process.poll() is None:  # 如果服务进程仍在运行
    process.terminate()  # 发送终止信号
    process.wait(timeout=10)  # 等待进程结束
    print("SGLang 服务已停止")  # 打印停止提示
else:  # 如果进程已结束
    print("SGLang 服务未在运行")  # 打印提示

# ---------- 清理 GPU 缓存 ----------
try:  # 尝试清理 GPU 缓存
    import torch  # 导入 PyTorch
    if torch.cuda.is_available():  # 如果 CUDA 可用
        torch.cuda.empty_cache()  # 清空 GPU 缓存
        print("GPU 缓存已清理")  # 打印清理完成提示
except ImportError:  # 如果 PyTorch 未安装
    pass  # 跳过清理

print("\n清理完成")  # 打印清理完成信息

## 常见问题排查

### 问题 1：优化参数后仍然 OOM

**现象：** 即使设置了 `--mem-fraction-static 0.5` 和 `--context-length 2048`，仍然报 `CUDA out of memory`。

**排查步骤：**
1. 确认模型大小是否超出 GPU 总显存（权重本身就放不下）
2. 检查是否有其他程序占用显存（如其他模型实例、Jupyter 占用的 PyTorch 内存）
3. 尝试更小的模型或量化模型
4. 重启 Jupyter Kernel 释放 PyTorch 占用的显存

### 问题 2：降低上下文长度后性能下降

**现象：** 设置 `--context-length 2048` 后，模型输出质量变差或回答被截断。

**排查步骤：**
1. 这是预期的权衡——更短的上下文意味着模型能「看到」的内容更少
2. 根据实际需求选择合适的上下文长度（不要盲目追求最小值）
3. 如果任务需要长上下文，考虑升级 GPU 或使用量化模型来节省显存
4. 对于多轮对话，可定期清理历史消息，保留最近的对话内容

---

### 结语

通过合理组合以上四种优化技巧，即使是 6GB 显存的消费级 GPU 也能顺利运行 0.5B-1B 级别的语言模型。关键是根据实际使用场景，在显存占用和性能之间找到最佳平衡点。